# 초구 예측 — model_18 (3단계: 위치 높이를 절대 높이 cm로, 독립 가정 3)

- 0장 0-1: 같은 투수×구종×손 안에서 실제 높이는 타자 존 아랫선·존 높이와 거의 무관 (계수 −0.01 · −0.04, 비율 가정이면 1 · 0.43), 퍼짐도 존 높이와 무관 (SD 약 28cm)
- 0장 0-2: model_13 위치를 절대 높이로 다시 계산하면 2019 24칸 +15.1 ×1000 (95% +11.9 ~ +18.2), 파라미터는 그대로
- 모델: model_16에서 **위치 높이 표현만** 존 비율(`z_norm`) -> 절대 높이로 (사용자 결정)
  - zn = (높이 cm − 평균 아랫선) / 평균 존 높이 (2017–18 값, 상수) -> 되돌릴 때도 같은 상수 (타자 존으로 늘이거나 줄이지 않음)
  - 초구·모든 카운트 공 모두 같은 표현 (높이 모멘트, 다른 카운트 잔차, 투수 안 분산)
  - 24칸 칸 경계는 그대로 그 타자 실제 존
  - 위치 파라미터(당김·반감기·모든 카운트·분산 따로)는 model_13 값 그대로 (다시 고르지 않음, 3단계 마감 때 다시 고르기)
  - 상황 보정(월 단위, 파라미터 그대로)은 새 위치 확률을 출발점으로 다시 학습
- 비교 대상: model_16

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys
if not os.path.exists("fp_common.py"):  # model/ 폴더에서 열면 한 단계 위(데이터·fp_common·results 있는 곳)로
    os.chdir("..")
sys.path.insert(0, os.getcwd())  # fp_common을 찾도록
import numpy as np
import pandas as pd
import fp_common as fc

MODEL = "model_18"

m, names = fc.load_data()
fp, y_type, y_zone, val, test = fc.first_pitches(m)
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]
print(f"초구 {len(fp):,}개 (검증 {val.sum():,} / 평가 {test.sum():,})")
import lightgbm as lgb
from scipy.special import log_softmax
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

초구 50,709개 (검증 13,458 / 평가 12,615)


## 0. 모델 전 확인

### 0-1. 데이터: 높이가 타자 존 비율대로 움직이나

- 같은 투수×구종×손 안에서 (그 그룹 평균을 빼고) 실제 높이 cm를 존 아랫선·존 높이의 차이로 회귀: 높이 − 그룹 평균 = a·(아랫선 − 평균) + b·(존 높이 − 평균)
  - 비율 가정이 맞으면 a ≈ 1, b ≈ 평균 z_norm (약 0.4) / 절대 높이면 a ≈ b ≈ 0
- 퍼짐: 같은 그룹 안 높이 SD를 타자 존 높이 3등분별로 -> 비율 가정이면 SD가 존 높이에 비례
- 초구 2017–2019 (2020 안 봄), 구종 의심 제외

In [2]:
d = fp[fp["GYEAR"].le(fc.VAL_YEAR)].copy()
d["z_cm"], d["h"], d["bot"] = d["CROSS_PLATE_Z_CM"], d["TOP_SZ_CM"] - d["BOTTOM_SZ_CM"], d["BOTTOM_SZ_CM"]
G3 = ["pitcher_code", "pitch_type", "same_side"]
for c in ["z_cm", "h", "bot"]:
    d[c + "_d"] = d[c] - d.groupby(G3)[c].transform("mean")  # 그룹 평균 빼기
X = d[["bot_d", "h_d"]].to_numpy()
coef, *_ = np.linalg.lstsq(X, d["z_cm_d"].to_numpy(), rcond=None)
resid = d["z_cm_d"].to_numpy() - X @ coef
se = np.sqrt(np.diag(np.linalg.inv(X.T @ X)) * resid.var())
print(f"초구 {len(d):,}개: 높이 차이 = {coef[0]:.3f}·아랫선 차이 (SE {se[0]:.3f}) + {coef[1]:.3f}·존 높이 차이 (SE {se[1]:.3f})")
print(f"비율 가정이면: 아랫선 1, 존 높이 {d['zn'].mean():.3f} (평균 z_norm) / 절대 높이면 둘 다 0")
print(f"타자 존 높이: 평균 {d['h'].mean():.1f}cm, SD {d['h'].std():.1f}cm, 아랫선 SD {d['bot'].std():.1f}cm / 그룹 안 높이 SD {d['z_cm_d'].std():.1f}cm")
d["h3"] = pd.qcut(d["h"], 3, labels=["존 낮은 타자", "중간", "존 높은 타자"])
t = d.groupby("h3", observed=True).agg(공=("h", "size"), 존높이=("h", "mean"), 아랫선=("bot", "mean"), 그룹안_높이SD=("z_cm_d", "std"), 그룹안_zn_SD=("zn", lambda s: (s - d.loc[s.index].groupby(G3)["zn"].transform("mean")).std()))
t["SD / 존높이"] = t["그룹안_높이SD"] / t["존높이"]
display(t.round(3))

초구 38,094개: 높이 차이 = -0.010·아랫선 차이 (SE 0.035) + -0.037·존 높이 차이 (SE 0.045)
비율 가정이면: 아랫선 1, 존 높이 0.430 (평균 z_norm) / 절대 높이면 둘 다 0
타자 존 높이: 평균 55.8cm, SD 3.3cm, 아랫선 SD 4.2cm / 그룹 안 높이 SD 28.1cm


,공,존높이,아랫선,그룹안_높이SD,그룹안_zn_SD,SD / 존높이
h3,,,,,,
존 낮은 타자,12738,52.382,50.462,27.782,0.531,0.530
중간,15213,56.021,49.502,28.327,0.506,0.506
존 높은 타자,10143,59.673,50.440,28.227,0.472,0.473


#### 🔍 0-1 결과: 높이는 타자 존 비율대로 움직이지 않음 (거의 절대 높이)
- 같은 투수×구종×손 안에서 높이 차이 = −0.010 · 아랫선 차이 (SE 0.035) − 0.037 · 존 높이 차이 (SE 0.045)
  - 비율 가정이면 1 · 0.43이어야 하는데 둘 다 0 근처 -> 투수는 타자 존과 상관없이 비슷한 cm 높이로 던짐
- 퍼짐도 그대로: 그룹 안 높이 SD가 존 낮은 타자 27.8cm · 중간 28.3cm · 존 높은 타자 28.2cm로 같음 -> z_norm 단위로는 0.53 · 0.51 · 0.47로 달라짐 (비율 가정이 틀려서)
- 타자 존 높이 자체의 차이가 작음 (SD 3.3cm, 아랫선 SD 4.2cm) — 그룹 안 높이 SD 28cm에 비하면 작아서 투수가 맞출 필요가 적거나, 존 높이 값(트래킹 추정)에 잡음이 있을 수 있음


### 0-2. 높이 표현 세 가지로 model_13 위치 다시 계산 (2019)

- 높이를 모으는 방식만 바꾸고 나머지(당김·반감기·모든 카운트·분산 따로 등)는 model_13 값 그대로
  - **A 존 비율** (지금): zn = (높이 − 아랫선) / 존 높이 -> 되돌릴 때 그 타자 아랫선 + zn × 그 타자 존 높이
  - **B 존 가운데 기준 cm**: zn = (높이 − 존 가운데) / H̄ + 0.5 -> 그 타자 존 가운데로만 옮기고 크기는 고정 (H̄ = 지난 시즌까지 평균 존 높이)
  - **C 절대 높이 cm**: zn = (높이 − 평균 아랫선) / H̄ -> 타자와 상관없음
  - 24칸 칸 경계는 어느 표현이든 그 타자 실제 존 기준 (정답은 같음)
- 초구·모든 카운트 공 모두 같은 표현으로 바꿔 쌓음 (높이 모멘트, 다른 카운트 잔차, 투수 안 분산)
- 비교: 2019 24칸 log loss·좌표 NLL (실제 구종 기준), 타자 존 높이 3등분별

In [3]:
LOC_REF, LOC_REF_HL = "decay", 173.0  # model_13 위치 정규분포
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0
LOC_SEASON_F = 0.707
LOC_AC = (None, 12.5, 17.3)
LOC_VAR = (200.0, 9.74, False, 316.0)
ZONE_M = 32
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = fc.gauss(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)
    return zone_probs(g, qt, cov), g



LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
pp17 = fc.load_results(["model_13"])["model_13"]
past = fp["GYEAR"] < fc.VAL_YEAR
H_BAR = float((fp.loc[past, "TOP_SZ_CM"] - fp.loc[past, "BOTTOM_SZ_CM"]).mean())  # 평균 존 높이 (지난 시즌까지)
BOT_BAR = float(fp.loc[past, "BOTTOM_SZ_CM"].mean())  # 평균 아랫선
print(f"H̄ = {H_BAR:.1f}cm, 평균 아랫선 = {BOT_BAR:.1f}cm")

def zn_of(z, top, bot, rep):  # 높이 cm -> 모을 값
    if rep == "A":
        return (z - bot) / (top - bot)
    if rep == "B":
        return (z - (top + bot) / 2) / H_BAR + 0.5
    return (z - BOT_BAR) / H_BAR

def set_rep(rep):  # fp·MO의 높이 열을 그 표현으로 바꾸고 캐시 비움
    fp["zn"] = zn_of(fp["CROSS_PLATE_Z_CM"], fp["TOP_SZ_CM"], fp["BOTTOM_SZ_CM"], rep)
    fp["zz"], fp["xz"] = fp["zn"] ** 2, fp["x"] * fp["zn"]
    MO["zn"] = zn_of(MO["CROSS_PLATE_Z_CM"], MO["TOP_SZ_CM"], MO["BOTTOM_SZ_CM"], rep)
    for cache in (fc._TABLES, _MO_RES, _MO_MOM, _WT):
        cache.clear()

def gauss_rep(M, q, rep):  # 모멘트 -> 정규분포 (그 표현으로 되돌림)
    mx, mz, xx, zz, xz = M[:, :5].T
    top, bot = q["TOP_SZ_CM"].to_numpy(), q["BOTTOM_SZ_CM"].to_numpy()
    if rep == "A":
        h, z0 = top - bot, bot
    elif rep == "B":
        h, z0 = np.full(len(q), H_BAR), (top + bot) / 2 - 0.5 * H_BAR
    else:
        h, z0 = np.full(len(q), H_BAR), np.full(len(q), BOT_BAR)
    vx = np.maximum(xx - mx ** 2, 4.0)
    vz = np.maximum(zz - mz ** 2, 1e-3) * h ** 2
    c = np.clip((xz - mx * mz) * h, -0.99 * np.sqrt(vx * vz), 0.99 * np.sqrt(vx * vz))
    return mx, z0 + mz * h, vx, vz, c

q19 = fp[val]
i9, yz9 = np.arange(len(q19)), y_zone[val]
h19 = (q19["TOP_SZ_CM"] - q19["BOTTOM_SZ_CM"]).to_numpy()
h3 = pd.qcut(h19, 3, labels=["존 낮은 타자", "중간", "존 높은 타자"])
RES = {}
for rep in ["A", "B", "C"]:
    set_rep(rep)
    qt = fp[val]
    M = loc_online(qt, fc.VAL_YEAR, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, LOC_AC, LOC_VAR)
    g = gauss_rep(M, qt, rep)
    P = fc.zone_from_gauss_cov(g, qt, ZONE_M)
    ll = -np.log(np.clip(P[i9, yz9], 1e-12, None))
    if rep == "A":
        assert np.allclose(ll, fc.align(pp17, fp, fc.VAL_YEAR)["ll_zone_true"].to_numpy()), "A = model_13 재현 실패"
        print("A (존 비율) = model_13 (재현 확인)")
    RES[rep] = {"ll": ll, "coord": -fc.log_dens(g, qt), "sz": np.sqrt(g[3])}
set_rep("A")  # 원래대로

gid9 = q19["gameID"].to_numpy()
def game_ci(dd, B=1000):
    s = pd.DataFrame({"g": gid9, "d": dd}).groupby("g")["d"].agg(["sum", "count"])
    i = np.random.default_rng(0).integers(0, len(s), (B, len(s)))
    return np.percentile(s["sum"].to_numpy()[i].sum(1) / s["count"].to_numpy()[i].sum(1), [2.5, 97.5])

rows = []
for rep, nm in [("A", "A 존 비율 (지금)"), ("B", "B 존 가운데 기준 cm"), ("C", "C 절대 높이 cm")]:
    dd = RES["A"]["ll"] - RES[rep]["ll"]
    lo, hi = game_ci(dd)
    r = {"표현": nm, "24칸 ll": RES[rep]["ll"].mean(), "좌표 NLL": RES[rep]["coord"].mean(), "이득 ×1000 (A 대비)": dd.mean() * 1000,
         "95% 하한": lo * 1000, "95% 상한": hi * 1000, "예측 높이 SD cm": RES[rep]["sz"].mean()}
    for lab in ["존 낮은 타자", "중간", "존 높은 타자"]:
        k = np.asarray(h3) == lab
        r[f"이득 {lab}"] = dd[k].mean() * 1000
    rows.append(r)
display(pd.DataFrame(rows).set_index("표현").round(4))
print("타자 존 높이 3등분 경계 (cm):", np.round(np.quantile(h19, [1 / 3, 2 / 3]), 1))

H̄ = 55.8cm, 평균 아랫선 = 51.0cm
A (존 비율) = model_13 (재현 확인)


,24칸 ll,좌표 NLL,이득 ×1000 (A 대비),95% 하한,95% 상한,예측 높이 SD cm,이득 존 낮은 타자,이득 중간,이득 존 높은 타자
표현,,,,,,,,,
A 존 비율 (지금),2.8872,9.2413,0.0000,0.0000,0.0000,27.9966,0.0000,0.0000,0.0000
B 존 가운데 기준 cm,2.8856,9.2389,1.6169,0.4249,2.8326,27.7577,1.6395,0.8762,2.6083
C 절대 높이 cm,2.8721,9.2229,15.1034,11.9154,18.1628,27.4772,18.7645,11.6608,15.2672


타자 존 높이 3등분 경계 (cm): [54.3 57.2]


#### 🔍 0-2 결과: 절대 높이(C)가 지금(A)보다 크게 좋음
| 표현 (2019, 실제 구종) | 24칸 ll | 좌표 NLL | 이득 ×1000 (A 대비, 95%) | 존 낮은 / 중간 / 존 높은 타자 |
| --- | --- | --- | --- | --- |
| A 존 비율 (지금) | 2.8872 | 9.2413 | 0 | 0 / 0 / 0 |
| B 존 가운데 기준 cm | 2.8856 | 9.2389 | +1.62 (+0.42 ~ +2.83) | +1.6 / +0.9 / +2.6 |
| **C 절대 높이 cm** | **2.8721** | **9.2229** | **+15.10** (+11.92 ~ +18.16) | +18.8 / +11.7 / +15.3 |

- A = model_13 재현 확인
- C가 3단계에서 지금까지 본 어떤 개선보다 큼 (상황 보정 model_14 +3.7의 약 4배), 타자 존 높이 세 구간 모두 좋아짐
- 파라미터는 model_13 값 그대로 (2019로 고르지 않음) -> 고르는 몫으로 부풀려진 숫자가 아님
- 이유 (0-1): 실제 높이가 타자 존과 거의 무관한데 A는 존 높이에 맞춰 늘이고 줄임 -> 쓸데없는 흔들림, 특히 아랫선이 다른 타자에게 중심이 어긋남 (B도 존 가운데로 옮겨서 조금만 좋아짐)
- 24칸 칸 경계는 어느 표현이든 그 타자 실제 존 기준 -> C는 "공은 절대 높이로 오고, 존은 타자마다 다름"


## 모델

In [4]:
# 0장이 모델 변수 이름을 덮어쓰지 않았는지 대비: 설정 상수를 다시 정함
CLASSES, ZONES, MOM = fc.CLASSES, fc.ZONES, fc.MOM
q_val, q_test = fp[val], fp[test]


## 1. 파라미터

- 모두 model_16 값 고정, 바꾸는 것: 위치 높이 표현 `LOC_HEIGHT = "abs"` 하나

In [5]:
# ① 구종 출발점 B4: model_06 값 (고정) - 반감기·alpha·hand_k, 시즌 초 배수 (기준점 = 지난 시즌 전부)
B4_P = dict(half_life=35.4, alpha=7.5, hand_k=5.48, season_f=0.25)
# ① LightGBM 변수: A 이번 타석 상황 + 앞 타석 초구 결과 원-핫 (fc.FIRST_CATS)
FEAT = fc.A_COLS + ["prev_first_" + c for c in fc.FIRST_CATS]
# ② 위치 정규분포: model_13 값 (고정)
LOC_REF, LOC_REF_HL = "decay", 173.0  # 기준점: 분석 대상 투수 구종×손, 전날까지 반감 173구
LOC_HL, K_TYPE, K_HAND = 35.4, 1000.0, 30.0  # 투수 기록 반감기, 투수×구종 당김 k, 투수×구종×손 당김 k
LOC_SEASON_F = 0.707  # 시즌 초 배수
LOC_AC = (None, 12.5, 17.3)  # 모든 카운트 위치 성향 (잔차 기준 반감 None = 누적, δ 반감, δ 당김 k)
LOC_VAR = (200.0, 9.74, False, 316.0)  # 분산 따로 (다른 카운트 퍼짐 반감, k_var, unc, 투수 안 분산 당김 k0)
# ② 위치 상황 보정 (model_14): 24칸 LightGBM, 출발점 = model_13 위치 log 확률
LOC_SIT = ["base1", "base2", "base3", "first_open", "sdiff", "inning", "out_cn", "home", "batorder"]  # 상황 변수 (0-3 묶음 4)
LOC_FEAT = CLASSES + ["same_side"] + LOC_SIT  # 구종 원-핫·타자 손 + 상황
LOC_TRAIN_FIRST = 2018  # 위치 보정 학습 첫 해 (2017은 model_13 위치 없음)
LOC_LGB, LOC_ROUNDS = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}, 168  # model_14 상황 보정 LightGBM (고정)
LOC_RETRAIN = "month"  # 상황 보정 재학습 주기 (model_16): 그 달 첫 경기 전날까지 데이터로 다시 학습
LOC_HEIGHT = "abs"  # 위치 높이 표현 (model_18): 절대 높이 cm (지금까지는 존 비율 z_norm)
ZONE_M = 32
# ① LightGBM: model_07 값 (고정)
LGB_P = {"num_leaves": 7, "min_data_in_leaf": 20, "lambda_l2": 1000.0}
LGB_ROUNDS = 730
RETRAIN = "month"  # 재학습 주기: 그 달 첫 경기 전날까지 데이터로 다시 학습 (3월은 4월에, 10월은 9월에 합침)
# 구종 보정 학습 해
TRAIN_FIRST = 2017

## 2. ① 구종: B4 + LightGBM 월 단위 재학습 (model_09과 같음)

### 2-1. B4 (model_06·07·09과 같음)

In [6]:
def b4_online(q, year, half_life, alpha, hand_k, season_f):  # model_06 B4 (n, 9): 지난 시즌 리그 비율 -> 투수 (alpha) -> 투수×손 (hand_k), 시즌 초 배수
    first = year == fp["GYEAR"].min()  # 첫 시즌(2017): 지난 기록 없음 -> 그 시즌 비율 (학습 데이터 출발점용)
    ref = fp.loc[fp["GYEAR"] == year if first else fp["GYEAR"] < year, CLASSES].mean().to_numpy()
    out = []
    for keys in (["pitcher_code"], ["pitcher_code", "same_side"]):
        S, N = fc.lookup(fc.prior_table(fp, keys, CLASSES, half_life, season_f=season_f), q, keys, CLASSES, season_f=season_f)
        out += [S, N[:, :1]]
    S, N, S2, N2 = out
    return fc.shrink(S2, N2, fc.shrink(S, N, ref, alpha), hand_k)

K = len(CLASSES)
P_B4 = np.full((len(fp), K), np.nan)  # 2017–2020
for yr in range(TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    P_B4[k] = b4_online(fp[k], yr, **B4_P)
b4_19 = fc.prob_metrics(P_B4[val], y_type[val])["logloss"]
assert abs(b4_19 - 1.09367) < 1e-5, b4_19  # model_06·07 B4의 2019 log loss
print(f"2019 B4 log loss {b4_19:.5f} = model_06·07 B4 (재현 확인)")
m07 = pd.read_parquet("results/model_07/per_pitch.parquet")  # 첫 달 재현 확인용
m09 = pd.read_parquet("results/model_09/per_pitch.parquet")  # 구종 재현 확인용 (구종은 model_09 그대로)
m13 = pd.read_parquet("results/model_13/per_pitch.parquet")  # 위치 정규분포 재현 확인용
m14 = pd.read_parquet("results/model_14/per_pitch.parquet")  # 참고
m16 = pd.read_parquet("results/model_16/per_pitch.parquet")  # 비교 대상

2019 B4 log loss 1.09367 = model_06·07 B4 (재현 확인)


### 2-2. 변수 (model_07과 같음)

- A 이번 타석 상황 + 앞 타석 초구 결과, 미래 정보 검사 다시 통과 확인

In [7]:
X_all = pd.concat([fc.situation_features(m, fp), fc.game_flow_features(m, fp)], axis=1)[FEAT]
assert X_all.notna().all().all()
for fn in (fc.situation_features, fc.game_flow_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(TRAIN_FIRST)], n=150)
    assert bad.empty, bad
display(X_all.describe().T[["mean", "min", "max"]].round(3))

미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


,mean,min,max
pc,33.394,0.0,124.0
tto,1.592,1.0,5.0
starter,0.690,0.0,1.0
inning,4.548,1.0,12.0
out_cn,0.970,0.0,2.0
base1,0.359,0.0,1.0
base2,0.194,0.0,1.0
base3,0.100,0.0,1.0
batorder,4.811,1.0,9.0
sdiff,0.369,-21.0,19.0


### 2-2. 월 단위 재학습과 구종 확률 (model_09과 같음)

- 확인: 구종 확률 = model_09 결과 파일 (재현)

In [8]:
y_all = fp["pitch_type"].map(fc.TYPE_IDX).to_numpy()
INIT = np.log(np.clip(P_B4, 1e-6, None))
INIT -= INIT.mean(1, keepdims=True)  # 출발점 (공마다 상수는 softmax에 영향 없음)
BASE = dict(objective="multiclass", num_class=K, learning_rate=0.03, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
            verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)  # model_05–07과 같음
yr_all = fp["GYEAR"].to_numpy()
DATE = pd.to_datetime(fp["gameID"].str[:8]).to_numpy()
MON = np.clip(fp["gameID"].str[4:6].astype(int).to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 LightGBM (트리 수 고정)
    return lgb.train({**BASE, **LGB_P}, lgb.Dataset(X_all.iloc[tr], y_all[tr], init_score=INIT[tr]), LGB_ROUNDS)

def type_prob(bst, idx):  # B4 출발점 + 트리 보정 -> 구종 확률 (n, 9)
    return np.exp(log_softmax(bst.predict(X_all.iloc[idx], raw_score=True) + INIT[idx], axis=1))

P_TYPE_ALL = np.full((len(fp), K), np.nan)
BST, plan = {}, []
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_all == yr)
    for mon in np.unique(MON[iy]):
        te = iy[MON[iy] == mon]
        start = DATE[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_all >= TRAIN_FIRST) & (DATE < start))
        assert DATE[tr].max() < start  # 미래 정보 없음
        BST[(yr, mon)] = bst = fit_rows(tr)
        P_TYPE_ALL[te] = type_prob(bst, te)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_all[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))
P11 = {yr: fc.align(m07, fp, yr)[fc.P_TYPE].to_numpy() for yr in (fc.VAL_YEAR, fc.TEST_YEAR)}  # model_07 구종 확률 (fp 그 해 순서)
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:  # 첫 달 = 시즌 단위 (model_07 재현)
    k = MON[msk] == MON[msk].min()
    assert np.allclose(P_TYPE_ALL[msk][k], P11[yr][k], atol=1e-6)
print("첫 달 구종 확률 = model_07 (재현 확인)")
rows = {}
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    for nm, P in [("B4", P_B4[msk]), ("model_07", P11[yr]), (MODEL, P_TYPE_ALL[msk])]:
        r = fc.prob_metrics(P, y_type[msk])
        rows[(yr, nm)] = {"log loss": r["logloss"], "top1": r["top1"], "top2": r["top2"]}
display(pd.DataFrame(rows).T.round(4))
for yr, msk in [(fc.VAL_YEAR, val), (fc.TEST_YEAR, test)]:
    assert np.allclose(P_TYPE_ALL[msk], fc.align(m09, fp, yr)[fc.P_TYPE].to_numpy(), atol=1e-6), f"{yr} 구종 확률이 model_09과 다름"
print("구종 확률 = model_09 (재현 확인)")

예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  24636           0  2018-10-14  2019-03-23
     5   2403  27204        2568  2019-04-30  2019-05-01
     6   2493  29607        4971  2019-05-31  2019-06-01
     7   1791  32100        7464  2019-06-30  2019-07-02
     8   2362  33891        9255  2019-07-31  2019-08-01
     9   1841  36253       11617  2019-08-31  2019-09-01
2020 5   2264  38094           0  2019-10-01  2020-05-05
     6   2379  40358        2264  2020-05-31  2020-06-02
     7   2233  42737        4643  2020-06-30  2020-07-01
     8   1985  44970        6876  2020-07-31  2020-08-01
     9   3754  46955        8861  2020-08-30  2020-09-01

첫 달 구종 확률 = model_07 (재현 확인)


log loss    top1    top2
2019 B4          1.0937  0.5279  0.8159
     model_07    1.0621  0.5380  0.8174
     model_18    1.0613  0.5385  0.8177
2020 B4          1.0916  0.5264  0.8179
     model_07    1.0570  0.5340  0.8209
     model_18    1.0561  0.5332  0.8218

구종 확률 = model_09 (재현 확인)


## 3. ② 위치: 절대 높이 정규분포 + 상황 보정 (월 단위)

### 3-1. 위치 정규분포 (높이 표현만 절대 높이로)

- fp·MO의 높이 열 `zn`을 절대 높이 표현으로 바꾸고 누적표 캐시를 비운 뒤 model_13 함수 그대로 계산
- `gauss_abs`: 모멘트 -> 정규분포, 높이 = 평균 아랫선 + zn × 평균 존 높이 (타자 존과 무관)
- 확인: 2019 실제 구종 기준 24칸이 0장 0-2의 C와 같은지

In [10]:
# 24칸 적분 함수: fc.zone_from_gauss_cov(g, q, m=32) (model_03부터)
def zone_probs(g, q, cov=True):  # 24칸 확률: cov=True 공분산 반영 (model_03) / False 독립 가정 (model_02까지)
    return fc.zone_from_gauss_cov(g, q, ZONE_M) if cov else fc.zone_from_gauss(g, q)

L_COLS = MOM + ZONES  # 당길 열: 위치 모멘트 5개(x, 높이 비율, 제곱, 곱) + 24칸 원-핫
K1, K2 = ["pitcher_code", "pitch_type"], ["pitcher_code", "pitch_type", "same_side"]  # 투수×구종 / 투수×구종×손
KR = ["pitch_type", "same_side"]  # 기준점 그룹: 분석 대상 투수 구종×손

def loc_league(q, year, ref="all", ref_hl=None):  # 기준점 (n, 29): "all" 지난 시즌 전부 / "last" 직전 시즌만 / "decay" 전날까지 반감기 ref_hl(공 수) 가중
    past = fp[fp["GYEAR"] < year]  # 지난 시즌까지
    overall = past[L_COLS].mean().to_numpy()  # 그 구종×손 기록이 없을 때 쓰는 전체 평균 (지난 시즌까지)
    if ref == "decay":
        S, N = fc.lookup(fc.prior_table(fp, KR, L_COLS, ref_hl), q, KR, L_COLS)  # 구종×손별 전날까지 가중 합계, 개수
        return np.where(N > 0, S / np.maximum(N, 1e-12), overall)  # 가중 평균, 기록 없으면 전체
    if ref == "last":
        past = fp[fp["GYEAR"] == year - 1]  # 직전 시즌만
    lg = past.groupby(KR)[L_COLS].mean()
    pr = lg.reindex(pd.MultiIndex.from_frame(q[KR]))  # 공마다 그 구종×손 행
    return pr.fillna(pd.Series(overall, index=L_COLS)).to_numpy()

def loc_sn(q, half_life, season_f, cache=True):  # 투수×구종, 투수×구종×손 전날까지 초구 위치 합계 S·개수 N 두 쌍 (시즌 초 배수 반영)
    return [fc.lookup(fc.prior_table(fp, keys, L_COLS, half_life, season_f=season_f, cache=cache), q, keys, L_COLS, season_f=season_f)
            for keys in (K1, K2)]

def shift_mean(M, dx, dz):  # 모멘트 M (n, 29)의 평균만 (dx, dz) 옮김, 분산·공분산 그대로 (24칸 원-핫 열은 정규분포에 안 쓰여 그대로)
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    nx, nz = mx + dx, mz + dz
    M[:, 2] += nx ** 2 - mx ** 2  # E[x²] = 분산 + 평균²
    M[:, 3] += nz ** 2 - mz ** 2
    M[:, 4] += nx * nz - mx * mz  # E[x·z] = 공분산 + 평균 곱
    M[:, 0], M[:, 1] = nx, nz
    return M

# 0-0 아닌 공 (구종 의심 제외): 모든 카운트 위치 성향 재료
MO = m[~m["label_doubt"] & ~(m["ball_before"].eq(0) & m["strike_before"].eq(0))].copy()
MO["x"], MO["zn"] = MO["plate_x_in_CM"], MO["z_norm"]  # 좌우 cm (타자 기준), 높이 비율 (존 아랫선 0, 윗선 1)
MO["cnt"] = MO["ball_before"].astype(int).astype(str) + "-" + MO["strike_before"].astype(int).astype(str)  # 카운트
CK = ["cnt", "pitch_type", "same_side"]  # 카운트 효과 그룹
_MO_RES = {}

def mo_resid(ref_hl):  # 0-0 아닌 공마다 위치 − 분석 대상 투수 그 카운트×구종×손 전날까지 평균 (반감 ref_hl구) -> 잔차 표 (rx 좌우 cm, rz 높이 비율)
    if ref_hl not in _MO_RES:
        S, N = fc.lookup(fc.prior_table(MO, CK, ["x", "zn"], ref_hl, cache=False), MO, CK, ["x", "zn"])
        ref = np.where(N > 0, S / np.maximum(N, 1e-12), np.nan)  # 그 카운트×구종×손 기록이 없으면 빈 값 (셈에서 빠짐)
        _MO_RES[ref_hl] = MO[K2 + ["game_date", "gameID", "pit_no"]].assign(rx=MO["x"].to_numpy() - ref[:, 0], rz=MO["zn"].to_numpy() - ref[:, 1])
    return _MO_RES[ref_hl]

def ac_sn(q, ref_hl, hl_b, season_f, cache=True):  # 공마다 투수×구종×손 다른 카운트 잔차 전날까지 합 S·개수 N (n, 2)
    d = mo_resid(ref_hl)
    return fc.lookup(fc.prior_table(d, K2, ["rx", "rz"], hl_b, season_f=season_f, cache=cache), q, K2, ["rx", "rz"], season_f=season_f)

MCOLS = ["x", "zn", "xx", "zz", "xz"]  # 위치 모멘트 (좌우, 높이 비율, 제곱, 곱)
_MO_MOM, _WT = {}, {}

def mo_mom(ref_hl):  # 다른 카운트 잔차 모멘트 표 (x = 좌우 잔차, zn = 높이 비율 잔차, 제곱·곱)
    if ref_hl not in _MO_MOM:
        d = mo_resid(ref_hl)
        _MO_MOM[ref_hl] = d.assign(xx=d["rx"] ** 2, zz=d["rz"] ** 2, xz=d["rx"] * d["rz"]).rename(columns={"rx": "x", "rz": "zn"})
    return _MO_MOM[ref_hl]

def within_tab(name, data, year):  # 구종×손별 투수 안 분산·공분산 (지난 시즌까지 투수×구종×손 분산의 (공 수 − 1) 가중 평균) 표와 전체 값
    if (name, year) not in _WT:
        past = data[data["game_date"].dt.year < year].dropna(subset=["x", "zn"])
        g = past.groupby(K2)
        n = g.size()
        w = (n - 1).clip(lower=0)
        c = pd.DataFrame({"vx": g["x"].var(), "vz": g["zn"].var(),
                          "cxz": (g["xz"].mean() - g["x"].mean() * g["zn"].mean()) * n / (n - 1).clip(lower=1)}).fillna(0)  # 표본 분산·공분산
        num = c.mul(w, axis=0).groupby(level=["pitch_type", "same_side"]).sum()
        wg = w.groupby(level=["pitch_type", "same_side"]).sum()  # 구종×손별 자유도 합 (표본 크기)
        _WT[(name, year)] = (num.div(wg, axis=0), c.mul(w, axis=0).sum() / w.sum(), wg)
    return _WT[(name, year)]

def v0_of(q, name, data, year, k0=0.0):  # 공마다 그 구종×손 투수 안 분산·공분산 (n, 3): 자유도 합이 적으면 전체 구종 값 쪽으로 k0만큼 당김
    tab, allv, wg = within_tab(name, data, year)
    idx = pd.MultiIndex.from_frame(q[KR])
    a = allv.to_numpy()[None, :]  # 전체 구종 투수 안 분산
    v = tab.reindex(idx).to_numpy()
    n = wg.reindex(idx).fillna(0).to_numpy()[:, None]  # 그 구종×손 자유도 합
    v = np.where(np.isnan(v), a, v)
    return np.where(n + k0 > 0, (n * v + k0 * a) / np.maximum(n + k0, 1e-12), a)  # 예: 너클 초구 3구(자유도 2)면 거의 전체 값

def own_var(q, year, ref_hl, hl_var, season_f, k0=0.0, cache=True):  # 공마다 다른 카운트 퍼짐 -> 초구 단위로 바꾼 자기 분산·공분산 (n, 3), (가중) 공 수 (n,), V0 (n, 3)
    data = mo_mom(ref_hl)
    S, N = fc.lookup(fc.prior_table(data, K2, MCOLS, hl_var, season_f=season_f, cache=cache), q, K2, MCOLS, season_f=season_f)
    e = S / np.maximum(N, 1e-12)
    v = np.column_stack([e[:, 2] - e[:, 0] ** 2, e[:, 3] - e[:, 1] ** 2, e[:, 4] - e[:, 0] * e[:, 1]])  # 자기 잔차 분산·공분산
    V0 = v0_of(q, "초구", fp, year, k0)
    V0m = v0_of(q, ("다른 카운트", ref_hl), data, year, k0)
    r = np.sqrt(np.maximum(v[:, :2], 0) / V0m[:, :2])  # 좌우·높이 SD 비율 (자기 / 투수 안)
    v = np.column_stack([V0[:, 0] * r[:, 0] ** 2, V0[:, 1] * r[:, 1] ** 2, v[:, 2] / np.sqrt(V0m[:, 0] * V0m[:, 1]) * np.sqrt(V0[:, 0] * V0[:, 1])])
    n = N[:, 0]
    return np.where(n[:, None] > 0, np.nan_to_num(v), 0.0), n, V0

def apply_var(M, v, n, V0, k_var, unc, N2, k_hand):  # 모멘트 M의 분산·공분산을 따로 당긴 값으로 바꿈 (평균 그대로)
    V = (n[:, None] * v + k_var * V0) / (n[:, None] + k_var)  # 투수 안 분산 쪽으로 k_var만큼
    if unc:
        V = V * (1 + 1 / (N2[:, :1] + k_hand))  # 평균을 덜 알수록 넓힘
    M = M.copy()
    mx, mz = M[:, 0], M[:, 1]
    M[:, 2], M[:, 3], M[:, 4] = V[:, 0] + mx ** 2, V[:, 1] + mz ** 2, V[:, 2] + mx * mz  # 분산 -> 모멘트
    return M

def loc_online(q, year, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None):  # 위치 모멘트 (n, 29)
    P = loc_league(q, year, ref, ref_hl)  # 출발: 기준점
    if ac is not None:  # 모든 카운트 위치 성향 (model_12)
        S, N = ac_sn(q, ac[0], ac[1], season_f)
        d = S / (N + ac[2])
        P = shift_mean(P, d[:, 0], d[:, 1])
    (S1, N1), (S2, N2) = loc_sn(q, half_life, season_f)
    M = fc.shrink(S2, N2, fc.shrink(S1, N1, P, k_type), k_hand)  # 투수×구종 -> 투수×구종×손
    if var is not None:  # 분산 따로 (model_13): var = (hl_var, k_var, unc, k0)
        v, n, V0 = own_var(q, year, None if ac is None else ac[0], var[0], season_f, var[3])
        M = apply_var(M, v, n, V0, var[1], var[2], N2, k_hand)
    return M

def loc_given_type(q, year, t, half_life, k_type, k_hand, ref="all", ref_hl=None, season_f=1.0, ac=None, var=None, cov=True):  # 구종이 t일 때 24칸 확률 (n, 24)과 좌표 정규분포
    qt = fc.as_type(q, t)
    g = gauss_abs(loc_online(qt, year, half_life, k_type, k_hand, ref, ref_hl, season_f, ac, var), qt)  # 절대 높이 (model_18)
    return zone_probs(g, qt, cov), g


# 높이 표현 (model_18): 절대 높이 cm, 상수는 2017–18 값 (2019·2020 모두 같은 상수, 미래 정보 없음)
past_h = fp["GYEAR"] < fc.VAL_YEAR
H_BAR = float((fp.loc[past_h, "TOP_SZ_CM"] - fp.loc[past_h, "BOTTOM_SZ_CM"]).mean())  # 평균 존 높이
BOT_BAR = float(fp.loc[past_h, "BOTTOM_SZ_CM"].mean())  # 평균 아랫선
fp["zn"] = (fp["CROSS_PLATE_Z_CM"] - BOT_BAR) / H_BAR  # 높이 열을 절대 높이 표현으로 (구종은 이 열을 안 씀)
fp["zz"], fp["xz"] = fp["zn"] ** 2, fp["x"] * fp["zn"]
MO["zn"] = (MO["CROSS_PLATE_Z_CM"] - BOT_BAR) / H_BAR  # 다른 카운트 공도 같은 표현
for cache in (fc._TABLES, _MO_RES, _MO_MOM, _WT):
    cache.clear()  # 높이 열이 바뀌어 누적표를 새로 (prior_table 캐시는 id(data) 키라 같은 객체면 옛 표를 씀)

def gauss_abs(M, q):  # 모멘트 (x, zn, xx, zz, xz) -> 2차원 정규분포, 높이는 절대 높이로 되돌림 (타자 존 무관)
    mx, mz, xx, zz, xz = M[:, :5].T
    vx = np.maximum(xx - mx ** 2, 4.0)
    vz = np.maximum(zz - mz ** 2, 1e-3) * H_BAR ** 2
    c = np.clip((xz - mx * mz) * H_BAR, -0.99 * np.sqrt(vx * vz), 0.99 * np.sqrt(vx * vz))
    return mx, BOT_BAR + mz * H_BAR, vx, vz, c

LOC17 = dict(half_life=LOC_HL, k_type=K_TYPE, k_hand=K_HAND, ref=LOC_REF, ref_hl=LOC_REF_HL, season_f=LOC_SEASON_F, ac=LOC_AC, var=LOC_VAR)
PZ_TRUE = np.full((len(fp), len(ZONES)), np.nan)  # 실제 구종 기준 model_13 24칸 확률 (2018–2020)
for yr in range(LOC_TRAIN_FIRST, fc.TEST_YEAR + 1):
    k = fp["GYEAR"].eq(yr).to_numpy()
    PZ_TRUE[k] = loc_given_type(fp[k], yr, fp.loc[k, "pitch_type"].to_numpy(), **LOC17)[0]
ll_ = -np.log(np.clip(PZ_TRUE[val][np.arange(val.sum()), y_zone[val]], 1e-12, None))
assert np.allclose(ll_, RES["C"]["ll"]), "0장 0-2 C와 다름"
print(f"절대 높이 2019 24칸 (실제 구종) {ll_.mean():.5f} = 0장 C (재현 확인) / model_13 {fc.align(m13, fp, fc.VAL_YEAR)['ll_zone_true'].mean():.5f}")
print("2018 계산된 공:", int(np.isfinite(PZ_TRUE[fp["GYEAR"].eq(2018).to_numpy()]).all(1).sum()))

절대 높이 2019 24칸 (실제 구종) 2.87214 = 0장 C (재현 확인) / model_13 2.88724
2018 계산된 공: 12395


### 3-2. 상황 보정 월 단위 재학습 (model_16과 같음, 출발점만 새 위치)

In [11]:
from scipy.special import log_softmax
for fn in (fc.situation_features, fc.opponent_features):
    bad = fc.check_no_future(fn, m, fp[fp["GYEAR"].ge(LOC_TRAIN_FIRST)], n=150)
    assert bad.empty, bad
SIT_L = fc.situation_features(m, fp)
XLOC = pd.concat([fp[CLASSES + ["same_side"]], SIT_L, fc.opponent_features(m, fp)[["home"]]], axis=1)
XLOC["first_open"] = (((SIT_L["base2"] + SIT_L["base3"]) > 0) & SIT_L["base1"].eq(0)).astype(int)  # 득점권인데 1루 비었음
XLOC = XLOC[LOC_FEAT]
NZ = len(ZONES)
INIT_L = np.log(np.clip(PZ_TRUE, 1e-6, None))
INIT_L -= INIT_L.mean(1, keepdims=True)  # 출발점
yr_l = fp["GYEAR"].to_numpy()
BASE_L = dict(objective="multiclass", num_class=NZ, learning_rate=0.05, feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1,
              verbose=-1, seed=0, deterministic=True, force_row_wise=True, num_threads=4)


DATE_L = fp["game_date"].to_numpy()
MON_L = np.clip(fp["game_date"].dt.month.to_numpy(), 4, 9)  # 재학습 묶음 달: 3월 -> 4월, 10월 -> 9월

def fit_rows(tr):  # 학습 행 tr로 상황 보정 LightGBM (model_14 파라미터, 트리 168)
    return lgb.train({**BASE_L, **LOC_LGB}, lgb.Dataset(XLOC.iloc[tr], y_zone[tr], init_score=INIT_L[tr]), LOC_ROUNDS)

BST_L, plan = {}, []  # (연도, 달) -> 그 달 예측용 모델
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    iy = np.flatnonzero(yr_l == yr)
    for mon in np.unique(MON_L[iy]):
        te = iy[MON_L[iy] == mon]
        start = DATE_L[te].min()  # 그 달 첫 경기일
        tr = np.flatnonzero((yr_l >= LOC_TRAIN_FIRST) & (DATE_L < start))
        assert DATE_L[tr].max() < start  # 미래 정보 없음
        BST_L[(yr, mon)] = fit_rows(tr)
        plan.append({"연도": yr, "월": mon, "예측 초구": len(te), "학습 초구": len(tr), "그 시즌 학습 초구": int((yr_l[tr] == yr).sum()),
                     "학습 끝": str(pd.Timestamp(DATE_L[tr].max()).date()), "예측 시작": str(pd.Timestamp(start).date())})
display(pd.DataFrame(plan).set_index(["연도", "월"]))

def loc_correct(P, q, year, t=None):  # model_13 24칸 확률 P (n, 24) -> 상황 보정 확률 (공마다 그 달 모델), t = 가정한 구종 (None이면 q의 실제 구종)
    X = XLOC.loc[q.index].copy()
    if t is not None:
        X.loc[:, CLASSES] = np.tile((np.array(CLASSES) == t).astype(float), (len(X), 1))  # 구종 원-핫을 t로
    init = np.log(np.clip(P, 1e-6, None))
    init -= init.mean(1, keepdims=True)
    mon = MON_L[fp.index.get_indexer(q.index)]
    out = np.empty_like(P)
    for mo in np.unique(mon):
        k = mon == mo
        out[k] = np.exp(log_softmax(BST_L[(year, mo)].predict(X[k], raw_score=True) + init[k], axis=1))
    return out



미래 정보 검사: 150개 중 다른 공 0개
미래 정보 검사: 150개 중 다른 공 0개


예측 초구  학습 초구  그 시즌 학습 초구        학습 끝       예측 시작
연도   월                                                  
2019 4   2568  12395           0  2018-10-14  2019-03-23
     5   2403  14963        2568  2019-04-30  2019-05-01
     6   2493  17366        4971  2019-05-31  2019-06-01
     7   1791  19859        7464  2019-06-30  2019-07-02
     8   2362  21650        9255  2019-07-31  2019-08-01
     9   1841  24012       11617  2019-08-31  2019-09-01
2020 5   2264  25853           0  2019-10-01  2020-05-05
     6   2379  28117        2264  2020-05-31  2020-06-02
     7   2233  30496        4643  2020-06-30  2020-07-01
     8   1985  32729        6876  2020-07-31  2020-08-01
     9   3754  34714        8861  2020-08-30  2020-09-01

### 3-3. 진단 (실제 구종 기준)

- model_16 대비 이득: 달별·주자·아웃 + 타자 존 높이 3등분별 (0장에서 세 구간 모두 좋아짐)
- 퍼짐 보정: 높이 표준화 잔차 SD, 80% 타원 (존 높이 3등분별)

In [12]:
def auc(p, o):  # Mann-Whitney AUC (같은 값은 평균 순위)
    r = pd.Series(p).rank().to_numpy()
    n1, n0 = o.sum(), len(o) - o.sum()
    return (r[o == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def runner3(q):
    return np.select([(q["base2"] + q["base3"]) > 0, q["base1"] > 0], ["득점권", "1루만"], "없음")

rows, cal = [], []
for yr, q, msk in [(fc.VAL_YEAR, q_val, val), (fc.TEST_YEAR, q_test, test)]:
    i, yz = np.arange(len(q)), y_zone[msk]
    P22 = loc_correct(PZ_TRUE[msk], q, yr)
    a20 = fc.align(m16, fp, yr)
    ll20, ll22 = a20["ll_zone_true"].to_numpy(), -np.log(np.clip(P22[i, yz], 1e-12, None))
    hz = (q["TOP_SZ_CM"] - q["BOTTOM_SZ_CM"]).to_numpy()
    h3 = np.asarray(pd.cut(hz, [0, 54.3, 57.2, 999], labels=["존 낮은 타자", "중간", "존 높은 타자"]))  # 0장 3등분 경계
    oc = (fc.ZONE_GROUP[yz] == "C").astype(float)
    d = pd.DataFrame({"월": MON_L[msk], "주자": runner3(SIT_L.loc[q.index]), "존 높이": h3, "이득": (ll20 - ll22) * 1000})
    for var in ["월", "주자", "존 높이"]:
        for v, g in d.groupby(var):
            rows.append({"연도": yr, "변수": var, "값": str(v), "공": len(g), "이득 ×1000 (model_16 대비)": g["이득"].mean()})
    rows.append({"연도": yr, "변수": "전체", "값": "", "공": len(d), "이득 ×1000 (model_16 대비)": d["이득"].mean(),
                 "존 밖 AUC": auc(P22[:, fc.ZONE_GROUP == "C"].sum(1), oc), "H/S/C 정확도": ((P22 @ fc.GROUP2).argmax(1) == pd.Series(fc.ZONE_GROUP[yz]).map({"H": 0, "S": 1, "C": 2}).to_numpy()).mean()})
    mx, mz, vx, vz, c = loc_given_type(q, yr, q["pitch_type"].to_numpy(), **LOC17)[1]
    dx, dz = q["x"].to_numpy() - mx, q["CROSS_PLATE_Z_CM"].to_numpy() - mz
    d2 = (vz * dx ** 2 - 2 * c * dx * dz + vx * dz ** 2) / (vx * vz - c ** 2)
    for lab in ["존 낮은 타자", "중간", "존 높은 타자", "전체"]:
        k = (h3 == lab) if lab != "전체" else np.ones(len(q), bool)
        cal.append({"연도": yr, "존 높이": lab, "평균 ez": (dz / np.sqrt(vz))[k].mean(), "SD ez": (dz / np.sqrt(vz))[k].std(), "80% 타원": (d2[k] <= -2 * np.log(0.2)).mean()})
display(pd.DataFrame(rows).set_index(["변수", "값", "연도"]).unstack("연도").round(4))
print("퍼짐 보정 (절대 높이, 실제 구종, 상황 보정 전 정규분포)")
display(pd.DataFrame(cal).set_index(["존 높이", "연도"]).unstack("연도").round(3))

공          이득 ×1000 (model_16 대비)          존 밖 AUC         H/S/C 정확도        
연도               2019     2020                   2019     2020    2019    2020      2019    2020
변수   값                                                                                          
월    4         2568.0      NaN                20.2830      NaN     NaN     NaN       NaN     NaN
     5         2403.0   2264.0                17.4054  12.1755     NaN     NaN       NaN     NaN
     6         2493.0   2379.0                 7.3901  15.6700     NaN     NaN       NaN     NaN
     7         1791.0   2233.0                10.8711  19.8876     NaN     NaN       NaN     NaN
     8         2362.0   1985.0                16.6940  13.9319     NaN     NaN       NaN     NaN
     9         1841.0   3754.0                10.1164   8.2054     NaN     NaN       NaN     NaN
전체            13458.0  12615.0                14.1077  13.2946  0.5637  0.5594     0.411  0.4076
존 높이 존 낮은 타자   4638.0   5407.0                17.0015  17.5959     NaN     NaN       NaN     NaN
     존 높은 타자   3711.0   1377.0                13.9459  26.6591     NaN     NaN       NaN     NaN
     중간        5109.0   5831.0                11.5982   6.1500     NaN     NaN       NaN     NaN
주자   1루만       2886.0   2723.0                11.5899  10.3316     NaN     NaN       NaN     NaN
     득점권       3346.0   3136.0                14.9940  11.8549     NaN     NaN       NaN     NaN
     없음        7226.0   6756.0                14.7028  15.1571     NaN     NaN       NaN     NaN

퍼짐 보정 (절대 높이, 실제 구종, 상황 보정 전 정규분포)


평균 ez         SD ez        80% 타원       
연도        2019   2020   2019   2020   2019   2020
존 높이                                             
전체       0.020 -0.013  1.000  1.005  0.814  0.806
존 낮은 타자 -0.016  0.007  0.992  1.015  0.819  0.809
존 높은 타자  0.018 -0.069  1.009  0.983  0.796  0.808
중간       0.054 -0.019  1.000  1.001  0.821  0.802

#### 🔍 3-3 결과: 두 해·모든 구간에서 고르게 좋아짐
- model_16 대비 이득 ×1000 (실제 구종): 2019 +14.1, 2020 +13.3 (두 해 거의 같음, 너클 영향 없음)
- 달별: 2019 +7.4 ~ +20.3, 2020 +8.2 ~ +19.9 (모든 달 +)
- 타자 존 높이별: 존 낮은 타자 +17.0 / +17.6, 중간 +11.6 / +6.2, 존 높은 타자 +13.9 / +26.7 -> 존이 평균에서 먼 타자일수록 큼 (비율로 늘이고 줄이던 몫이 컸던 곳)
- 주자별: 1루만 · 득점권 · 없음 모두 +10 ~ +15
- 퍼짐 보정 (상황 보정 전 정규분포): 높이 SD ez 1.00 / 1.01, 80% 타원 0.814 / 0.806, 존 높이 3등분 모두 0.80–0.82 -> 절대 높이로 바꿔도 퍼짐이 맞음
- 존 밖 AUC 0.564 / 0.559 (model_16 0.556 / 0.553)


## 4. 실전 예측 (예측 구종으로 위치 이어서)

- model_12과 같음, 위치만 3-1에서 고른 값 (분산 따로 포함)

In [13]:
def predict(q, year, Pt):  # 초구 q의 공별 예측·점수
    n, i = len(q), np.arange(len(q))
    yt, yz = q["pitch_type"].map(fc.TYPE_IDX).to_numpy(), q["zone"].map(fc.ZONE_IDX).to_numpy()  # 실제 구종·칸 번호
    nll = lambda p: -np.log(np.clip(p, 1e-12, None))  # 공별 log loss (prob_metrics와 같은 방식으로 자름)

    # ① 구종 확률 (n, 9): 2장 결과를 받음

    # ② 위치: 구종 9개 각각 가정 -> 24칸 확률, 실제 위치의 로그 밀도, 예측 중심 (좌우, 높이)
    PZ, LD, MX, MZ = [], [], [], []
    for t in CLASSES:
        P, g = loc_given_type(q, year, t, LOC_HL, K_TYPE, K_HAND, LOC_REF, LOC_REF_HL, LOC_SEASON_F, LOC_AC, LOC_VAR)
        P = loc_correct(P, q, year, t)  # 상황 보정 (model_14), 정규분포 g는 model_13 그대로
        PZ.append(P); LD.append(fc.log_dens(g, q)); MX.append(g[0]); MZ.append(g[1])
    PZ, LD, MX, MZ = np.stack(PZ, 1), np.stack(LD, 1), np.stack(MX, 1), np.stack(MZ, 1)  # (n, 구종, ...)
    joint = Pt[:, :, None] * PZ  # 구종 확률 × 그 구종일 때 칸 확률 = 구종×칸 결합 확률 (n, 9, 24)
    P_zone = joint.sum(1)  # 실전 24칸 확률 = 구종으로 합침
    # 좌표: 구종 확률로 섞은 정규분포에서 실제 위치의 로그 밀도
    ld_mix = np.log(np.einsum("nt,nt->n", Pt, np.exp(LD - LD.max(1, keepdims=True)))) + LD.max(1)  #  공마다 최댓값 빼기 -> exp -> 섞기 -> log (계산 안전)
    dist_true = np.hypot(q["x"] - MX[i, yt], q["CROSS_PLATE_Z_CM"] - MZ[i, yt])  # 실제 구종 중심과의 거리 cm
    dist_mix = np.hypot(q["x"] - (Pt * MX).sum(1), q["CROSS_PLATE_Z_CM"] - (Pt * MZ).sum(1))  # 구종 확률로 가중한 중심과의 거리 cm

    t_hat = np.array(CLASSES)[Pt.argmax(1)]  # 1등 구종 이름

    # 공별 결과 표 (필수: 키, 구종 확률 9개, 실전 24칸 확률 / 나머지는 선택 열)
    return pd.concat([
        q[["gameID", "pit_no", "pitcher_code", "GYEAR", "pitch_type", "zone"]].reset_index(drop=True),
        pd.DataFrame(Pt, columns=fc.P_TYPE),  # p_type_<구종>
        pd.DataFrame(P_zone, columns=fc.P_ZONE),  # p_zone_<칸>
        pd.DataFrame({"t_hat": t_hat}),  # 1등 구종 이름
        pd.DataFrame({
            "ll_type": nll(Pt[i, yt]),  # 구종 log loss
            "ll_zone": nll(P_zone[i, yz]),  # 24칸 log loss (실전)
            "ll_zone_true": nll(PZ[i, yt, yz]),  # 24칸 log loss (실제 구종)
            "ll_joint": nll(joint[i, yt, yz]),  # 구종×칸 log loss
            "nll_coord": -ld_mix,  # 좌표 NLL (실전)
            "nll_coord_true": -LD[i, yt],  # 좌표 NLL (실제 구종)
            "dist": np.asarray(dist_mix),  # 좌표 거리 cm (실전)
            "dist_true": np.asarray(dist_true),  # 좌표 거리 cm (실제 구종)
        })], axis=1)

In [14]:
# 2019 검증 + 2020 평가 초구 전부 예측
pp = pd.concat([predict(q_val, fc.VAL_YEAR, P_TYPE_ALL[val]), predict(q_test, fc.TEST_YEAR, P_TYPE_ALL[test])], ignore_index=True)
print(pp.shape)
# 확인: 구종은 model_16과 같아야 함 (위치는 24칸·좌표 모두 바뀜)
chk = pp.merge(m16, on=fc.KEYS, suffixes=("", "_16"))
same = ["ll_type"] + fc.P_TYPE
bad = [c for c in same if not np.allclose(chk[c], chk[c + "_16"], equal_nan=True, atol=1e-6)]
assert len(chk) == len(pp) and not bad, f"model_16과 다른 열: {bad}"
print("구종 = model_16 (재현 확인), 위치(24칸·좌표)만 바뀜")

(26073, 48)
구종 = model_16 (재현 확인), 위치(24칸·좌표)만 바뀜


## 5. 평가

- 채점은 `fc.evaluate`, model_16과 나란히 / 경기 단위 부트스트랩 (model_18 − model_16, 2020)
- 이번에는 좌표(정규분포)도 바뀜 -> 좌표 NLL·거리도 비교, 2020 너클 뺀 값도 봄

In [15]:
def eval_years(pp_):  # 2019·2020 주요 지표 표
    return pd.DataFrame({"검증 2019": fc.evaluate(pp_, fp, fc.VAL_YEAR), "평가 2020": fc.evaluate(pp_, fp, fc.TEST_YEAR)})

summary = eval_years(pp)
m16_sum = eval_years(m16)
display(pd.concat({MODEL: summary, "model_16": m16_sum, "차이 (model_16 대비)": summary - m16_sum}, axis=1).round(4))

model_18          model_16          차이 (model_16 대비)        
                     검증 2019  평가 2020  검증 2019  평가 2020          검증 2019 평가 2020
구종 logloss            1.0613   1.0561   1.0613   1.0561           0.0000  0.0000
구종 top1               0.5385   0.5332   0.5385   0.5332           0.0000  0.0000
구종 top2               0.8177   0.8218   0.8177   0.8218           0.0000  0.0000
구종 top3               0.9374   0.9412   0.9374   0.9412           0.0000  0.0000
구종 macro F1           0.3226   0.3545   0.3226   0.3545           0.0000  0.0000
24칸 logloss (실전)      2.9278   2.9266   2.9413   2.9402          -0.0135 -0.0135
24칸 top1              0.1384   0.1371   0.1329   0.1331           0.0056  0.0040
24칸 top3              0.3041   0.3066   0.2992   0.2999           0.0049  0.0067
H/S/C logloss         1.0723   1.0728   1.0733   1.0742          -0.0011 -0.0014
H/S/C 정확도             0.4019   0.4024   0.4010   0.3981           0.0010  0.0043
24칸 logloss (실제 구종)   2.8678   2.8703   2.8819   2.8836          -0.0141 -0.0133
구종×칸 logloss          3.9291   3.9264   3.9432   3.9397          -0.0141 -0.0133
좌표 평균거리 cm           32.3944  31.6319  32.6377  31.8198          -0.2433 -0.1880
좌표 NLL                9.2949   9.2493   9.3124   9.2662          -0.0175 -0.0169

In [16]:
display(fc.bootstrap_diff(fc.pitch_scores(m16, fp), fc.pitch_scores(pp, fp)).round(4))  # 음수 = 이 모델이 좋음 (b − a)
for yr in (fc.VAL_YEAR, fc.TEST_YEAR):
    a20, a22 = fc.align(m16, fp, yr), fc.align(pp, fp, yr)
    nk = a20["pitch_type"].ne("KNUC").to_numpy()
    d_true = a22["ll_zone_true"].to_numpy() - a20["ll_zone_true"].to_numpy()
    print(f"{yr} 24칸 실제 구종 차이 (model_18 − model_16): 전체 {d_true.mean():+.5f}, 너클 뺌 {d_true[nk].mean():+.5f}")

,차이 (b−a),95% 하한,95% 상한,판정
ll_type,0.0000,0.0000,0.0000,차이 불확실
ll_zone,-0.0135,-0.0161,-0.0110,좋아짐
ll_joint,-0.0133,-0.0161,-0.0106,좋아짐
nll_coord,-0.0169,-0.0197,-0.0141,좋아짐
dist,-0.1880,-0.2306,-0.1418,좋아짐


2019 24칸 실제 구종 차이 (model_18 − model_16): 전체 -0.01411, 너클 뺌 -0.01411
2020 24칸 실제 구종 차이 (model_18 − model_16): 전체 -0.01329, 너클 뺌 -0.01329


### 🥇1등🥇
#### 2020 24칸 log loss 실전 2.9402 -> 2.9266 (−0.0135), 위치 점수 4개 모두 부트스트랩 좋아짐 -> 채택
- 구종은 model_16과 같음 (재현 확인), 위치 높이 표현만 바꿈 (파라미터는 다시 고르지 않음)
- 2019 실제 구종 −0.0141 / 실전 −0.0135, 2020 실제 구종 −0.0133 (너클 빼도 같음) / 실전 −0.0135 -> 두 해 크기가 같음
- 부트스트랩 (2020, model_18 − model_16): 24칸 −0.0135 (95% −0.0161 ~ −0.0110), 구종×칸 −0.0133, 좌표 NLL −0.0169, 거리 −0.19cm 모두 좋아짐
- 24칸 top1 +0.0056 / +0.0040, top3 +0.0049 / +0.0067, H/S/C 정확도 +0.0010 / +0.0043 -> 개선 대부분은 같은 H/S/C 안에서 어느 칸인지 (H/S/C log loss는 −0.0011 / −0.0014)
- 원인 (0-1): 같은 투수×구종 안에서 공 높이는 타자 존과 거의 무관 -> 존 비율로 옮기고 늘이던 것이 오차, 존이 평균에서 먼 타자일수록 이득이 큼
- 1단계부터 쓴 z_norm 가정이 틀렸던 것 -> 위치 파라미터(1단계에서 이 가정 위에서 고름)를 3단계 마감 때 다시 고르는 게 중요
- **채택 -> 다음 비교 출발점** (사용자 결정 2026-11-11), 3단계에서 가장 큰 개선

## 6. 저장

- `results/model_18/` (채택하면 `compare.ipynb` `ADOPTED`에 추가)

In [17]:
fc.save_results(MODEL, pp, {"summary": summary})

저장: results\model_18 ['per_pitch.parquet', 'summary.csv']
